# vnoiser Pipeline

This notebook runs the denoising pipeline on an actual DS01 JEDI3sub `.mat` recording. It keeps the upstream plots diagnostic, then shows the final denoised trace and consolidated event calls only at the end.

In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import dendrogram
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.decomposition import PCA

from vnoiser import Denoiser, JediSub3Dataset
from vnoiser.denoiser import ClusteringConfig, cwtReducerConfig, thresConfig

In [2]:
duration_s = 5.0
seed = 21
dataset_root = next(
    (path for path in (Path("DS01-JEDI-sub3"), Path("../DS01-JEDI-sub3")) if path.exists()),
    None,
)
if dataset_root is None:
    raise FileNotFoundError("Expected DS01-JEDI-sub3/ next to the repo root or notebooks/.")

dataset = JediSub3Dataset(dataset_root)
recording = dataset.sample(duration_s=duration_s, seed=seed, require_events=True)
raw_trace = recording.trace.astype(float)
raw_std = np.std(raw_trace)
if raw_std == 0:
    raise ValueError("sampled raw trace has zero variance")

# The real DS01 traces used here have AP-associated positive excursions in fluo_mean.
# Use a robust center but keep the sign of the recorded trace.
denoiser_input = (raw_trace - np.median(raw_trace)) / raw_std

model = Denoiser(
    fs=recording.fs_hz,
    freq_scales=np.logspace(np.log10(2), np.log10(1800), num=64),
    lp_cutoff=20.0,
    fir_window_ms=80.0,
    cfg_clust=ClusteringConfig(n_components=16, n_comp_clu=8, n_clusters=5, n_subclusters=8),
    cfg_reducer=cwtReducerConfig(slow_upthres=1.5, fast_upthres=2.0),
    cfg_thres=thresConfig(thres_type="soft"),
)
denoised, raw_cluster_starts = model.run(denoiser_input)
denoised = np.real(denoised)

In [3]:
def robust_zscore(values, axis=None):
    values = np.asarray(values, dtype=float)
    median = np.median(values, axis=axis, keepdims=True)
    mad = 1.4826 * np.median(np.abs(values - median), axis=axis, keepdims=True)
    std = np.std(values, axis=axis, keepdims=True)
    scale = np.where(mad > 0, mad, np.where(std > 0, std, 1.0))
    return (values - median) / scale


def downsample(x, y, max_points=3500):
    step = max(1, int(np.ceil(len(x) / max_points)))
    return x[::step], y[::step]


def heatmap_grid(matrix, max_time=900, max_rows=64):
    row_step = max(1, int(np.ceil(matrix.shape[0] / max_rows)))
    col_step = max(1, int(np.ceil(matrix.shape[1] / max_time)))
    return row_step, col_step, matrix[::row_step, ::col_step]


def consolidated_pca_event_calls(
    masked_cluster_traces,
    reference_trace,
    fs_hz,
    threshold_z=3.0,
    prominence_z=1.0,
    min_distance_ms=6.0,
    smooth_ms=1.5,
):
    '''Return one final event call per PCA-score peak.

    The denoiser's raw event starts are per cluster, so the same real peak can
    appear several times. For the notebook's final calls, collapse the masked
    cluster traces into PC1 and peak-pick that single score.
    '''
    features = np.asarray(masked_cluster_traces, dtype=float)
    if features.ndim != 2 or features.shape[0] == 0 or features.shape[1] == 0:
        return np.array([], dtype=int), np.zeros_like(reference_trace), threshold_z

    feature_z = robust_zscore(features, axis=1)
    score = PCA(n_components=1).fit_transform(feature_z.T).ravel()
    corr = np.corrcoef(score, reference_trace)[0, 1] if np.std(score) > 0 else 0
    if np.isfinite(corr) and corr < 0:
        score *= -1

    sigma = max(1, int(round((smooth_ms / 1000.0) * fs_hz)))
    score_z = robust_zscore(gaussian_filter1d(score, sigma=sigma)).ravel()
    distance = max(1, int(round((min_distance_ms / 1000.0) * fs_hz)))
    peaks, _ = find_peaks(
        score_z,
        height=threshold_z,
        prominence=prominence_z,
        distance=distance,
    )
    return peaks.astype(int), score_z, threshold_z


coeff_power = np.abs(model.coeff_)
row_step, col_step, heat_power = heatmap_grid(coeff_power)
heat_t = recording.t[::col_step]
heat_freq = model.freqs_[::row_step]
reduced = model.reduced_["reduced_coeff"]
selected_freqs = model.reduced_["reduced_freqs"]
masks = model.threshold_result_["clu_label"]
masked_scaled = model.threshold_result_["all_scaled"]
event_items = list(model.reduced_["event_onsets"].items())
cluster_names = [name for name, _ in event_items]

pca_event_indices, pca_event_score_z, pca_threshold_z = consolidated_pca_event_calls(
    masked_scaled,
    denoiser_input,
    recording.fs_hz,
)
pca_event_times = pca_event_indices / recording.fs_hz

t_plot, input_plot = downsample(recording.t, denoiser_input)
_, denoised_plot = downsample(recording.t, denoised)
_, raw_plot = downsample(recording.t, raw_trace)
_, pca_score_plot = downsample(recording.t, pca_event_score_z)

print(
    f"{recording.path.name} | {len(recording.t):,} samples at {recording.fs_hz:.0f} Hz | "
    f"annotated APs {len(recording.events_ap_indices)} | "
    f"raw per-cluster starts {len(raw_cluster_starts)} | PCA calls {len(pca_event_indices)} | "
    f"CWT {model.coeff_.shape[0]} x {model.coeff_.shape[1]}"
)

VAttached_stan60_expt1_scan24_apical1_mini.mat | 50,000 samples at 10000 Hz | annotated APs 36 | raw per-cluster starts 391 | PCA calls 38 | CWT 64 x 50000


## Pipeline 1. Overview

This figure is the first pass sanity check: confirm the real trace window, the wavelet energy, the reduced cluster traces, and the adaptive masks all line up in time. It intentionally does not show the final denoised output.

In [4]:
overview = make_subplots(
    rows=2,
    cols=2,
    specs=[[{}, {}], [{}, {}]],
    subplot_titles=(
        "Pipeline 1a. Raw DS01 trace and z-scored input",
        "Pipeline 1b. CWT power by frequency",
        "Pipeline 1c. Reduced cluster representatives",
        "Pipeline 1d. Adaptive masks",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)
overview.add_trace(go.Scatter(x=t_plot, y=raw_plot, mode="lines", name="raw fluo_mean", line={"color": "#2f4858", "width": 1}), row=1, col=1)
overview.add_trace(go.Scatter(x=t_plot, y=input_plot, mode="lines", name="z-scored input", line={"color": "#4c78a8", "width": 1}), row=1, col=1)
overview.add_trace(go.Scatter(x=recording.events_ap_times_s, y=denoiser_input[recording.events_ap_indices], mode="markers", name="annotated AP", marker={"color": "#d7263d", "size": 5}), row=1, col=1)
overview.add_trace(go.Heatmap(x=heat_t, y=heat_freq, z=heat_power, colorscale="Viridis", colorbar={"title": "|CWT|", "len": 0.34}), row=1, col=2)
for i, band in enumerate(selected_freqs):
    x, y = downsample(recording.t, reduced[i])
    overview.add_trace(go.Scatter(x=x, y=y, mode="lines", name=f"{band[0]:.0f}-{band[1]:.0f} Hz", line={"width": 1}), row=2, col=1)
mask_step = max(1, int(np.ceil(masks.shape[1] / 900))) if masks.size else 1
overview.add_trace(go.Heatmap(x=recording.t[::mask_step], y=np.arange(masks.shape[0]) + 1, z=masks[:, ::mask_step], colorscale="Blues", showscale=False), row=2, col=2)
overview.update_xaxes(title_text="time (s)", row=1, col=1)
overview.update_xaxes(title_text="time (s)", row=1, col=2)
overview.update_xaxes(title_text="time (s)", row=2, col=1)
overview.update_xaxes(title_text="time (s)", row=2, col=2)
overview.update_yaxes(title_text="trace / z", row=1, col=1)
overview.update_yaxes(title_text="frequency (Hz)", type="log", row=1, col=2)
overview.update_yaxes(title_text="coefficient", row=2, col=1)
overview.update_yaxes(title_text="cluster", row=2, col=2)
overview.update_layout(
    title={"text": "Pipeline 1. Real Recording Through Wavelet Reduction", "x": 0.01, "xanchor": "left"},
    height=840,
    template="plotly_white",
    hovermode="x unified",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.03, "x": 0},
    margin={"t": 130, "b": 55, "l": 70, "r": 40},
)
overview.show()

**Pipeline 1a:** Use this to verify the sampled real recording window and AP annotations. The dark trace is raw `fluo_mean`; the blue trace is the z-scored input passed to `Denoiser`.

**Pipeline 1b:** Bright horizontal bands show where the CWT sees time-localized energy. APs should create high-frequency vertical structure; slower fluorescence drift or broad events should sit lower in frequency.

**Pipeline 1c:** Each line is one selected frequency-cluster representative after row averaging and smoothing. These traces are the inputs to the adaptive masks. Look for clusters that fire everywhere or never fire; those are candidates for parameter/pruning changes.

**Pipeline 1d:** Blue regions are the adaptive masks that decide which cluster activity survives reconstruction. Good masks should be sparse and aligned to real event structure, not uniformly on.

## Pipeline 2. Interactive Cluster Inspector

Use the cluster dropdown and time window slider to inspect one selected cluster at a time. This is where manual pruning decisions should happen before judging the final reconstruction.

In [5]:
cluster_options = [(name, i) for i, name in enumerate(cluster_names)]
if not cluster_options:
    raise ValueError("No event-bearing clusters were selected by WaveletReducer.")
cluster_dropdown = widgets.Dropdown(options=cluster_options, description="cluster")
time_slider = widgets.FloatRangeSlider(value=[0.0, duration_s], min=0.0, max=duration_s, step=0.05, continuous_update=False, description="window")

explorer = go.FigureWidget(make_subplots(
    rows=2,
    cols=2,
    specs=[[{}, {}], [{}, {}]],
    subplot_titles=(
        "Pipeline 2a. Wavelet power context",
        "Pipeline 2b. Selected cluster representative",
        "Pipeline 2c. Z-scored input context",
        "Pipeline 2d. Selected cluster mask",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
))
explorer.add_trace(go.Heatmap(x=heat_t, y=heat_freq, z=heat_power, colorscale="Viridis", colorbar={"title": "|CWT|", "len": 0.30}), row=1, col=1)
explorer.add_trace(go.Scatter(x=[], y=[], mode="lines", name="cluster coeff", line={"color": "#1b998b", "width": 1.2}), row=1, col=2)
explorer.add_trace(go.Scatter(x=t_plot, y=input_plot, mode="lines", name="z-scored input", line={"color": "#4c78a8", "width": 1}), row=2, col=1)
explorer.add_trace(go.Scatter(x=recording.events_ap_times_s, y=denoiser_input[recording.events_ap_indices], mode="markers", name="annotated AP", marker={"color": "#d7263d", "size": 5}), row=2, col=1)
explorer.add_trace(go.Scatter(x=[], y=[], mode="lines", fill="tozeroy", name="mask", line={"color": "#5b5f97", "width": 1.2}), row=2, col=2)
explorer.update_layout(
    title={"text": "Pipeline 2. Cluster-Level Inspection", "x": 0.01, "xanchor": "left"},
    height=780,
    template="plotly_white",
    hovermode="x unified",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.03, "x": 0},
    margin={"t": 130, "b": 55, "l": 70, "r": 40},
)
explorer.update_xaxes(title_text="time (s)", row=1, col=1)
explorer.update_xaxes(title_text="time (s)", row=1, col=2)
explorer.update_xaxes(title_text="time (s)", row=2, col=1)
explorer.update_xaxes(title_text="time (s)", row=2, col=2)
explorer.update_yaxes(title_text="Hz", type="log", row=1, col=1)
explorer.update_yaxes(title_text="coefficient", row=1, col=2)
explorer.update_yaxes(title_text="z", row=2, col=1)
explorer.update_yaxes(title_text="mask", row=2, col=2)


def update_explorer(*_):
    idx = cluster_dropdown.value
    lo, hi = time_slider.value
    keep = (recording.t >= lo) & (recording.t <= hi)
    t_sel = recording.t[keep]
    step = max(1, int(np.ceil(len(t_sel) / 2500))) if len(t_sel) else 1
    with explorer.batch_update():
        explorer.data[1].x = t_sel[::step]
        explorer.data[1].y = reduced[idx, keep][::step]
        explorer.data[4].x = t_sel[::step]
        explorer.data[4].y = masks[idx, keep][::step]
        explorer.layout.xaxis2.range = [lo, hi]
        explorer.layout.xaxis3.range = [lo, hi]
        explorer.layout.xaxis4.range = [lo, hi]

cluster_dropdown.observe(update_explorer, names="value")
time_slider.observe(update_explorer, names="value")
update_explorer()
display(widgets.VBox([widgets.HBox([cluster_dropdown, time_slider]), explorer]))

**Pipeline 2a:** This keeps the selected cluster in the context of the full CWT. Use it to ask whether the cluster belongs to a localized frequency band or sits in a noisy region.

**Pipeline 2b:** This is the actual reduced coefficient trace for the selected cluster. A useful cluster should have clear event-like excursions, not broad baseline wobble or continuous high activity.

**Pipeline 2c:** Use the raw z-scored input and AP annotations as the local reference while inspecting the selected cluster. The final denoised answer is held back until Pipeline 4.

**Pipeline 2d:** This is the selected cluster mask. If the mask is on during obvious noise, tighten thresholds; if it misses visible events, loosen thresholds or inspect adjacent clusters.

## Pipeline 3. Frequency Clustering

This dendrogram shows how CWT frequency rows were grouped before reduction. In a live Jupyter session, clicking tree segments prints the linkage height under the plot.

In [6]:
dendro = dendrogram(model.cluster_result_["Z"], no_plot=True)
tree = go.FigureWidget()
for xs, ys in zip(dendro["icoord"], dendro["dcoord"]):
    tree.add_trace(go.Scatter(
        x=xs,
        y=ys,
        mode="lines+markers",
        marker={"size": 4, "opacity": 0.25},
        line={"color": "#2f4858", "width": 1.5},
        hovertemplate="linkage height %{y:.3f}<extra>Pipeline 3a</extra>",
        showlegend=False,
    ))
leaf_x = np.arange(5, 10 * len(dendro["ivl"]) + 5, 10)
leaf_labels = [int(x) for x in dendro["ivl"]]
leaf_freqs = model.freqs_[leaf_labels]
tree.add_trace(go.Scatter(
    x=leaf_x,
    y=np.zeros_like(leaf_x),
    mode="markers",
    marker={"size": 6, "color": leaf_freqs, "colorscale": "Viridis", "colorbar": {"title": "Hz"}},
    text=[f"row {i}: {f:.1f} Hz" for i, f in zip(leaf_labels, leaf_freqs)],
    hovertemplate="%{text}<extra>frequency row</extra>",
    name="frequency rows",
))
tree.update_layout(
    title={"text": "Pipeline 3a. Frequency-Band Dendrogram", "x": 0.01, "xanchor": "left"},
    height=540,
    template="plotly_white",
    xaxis={"showticklabels": False, "title": "CWT frequency rows"},
    yaxis={"title": "linkage distance"},
    margin={"t": 80, "b": 55, "l": 70, "r": 40},
)
click_out = widgets.Output()


def tree_clicked(trace, points, state):
    if not points.point_inds:
        return
    with click_out:
        click_out.clear_output()
        i = points.point_inds[0]
        x = trace.x[i]
        y = trace.y[i]
        print(f"Pipeline 3a click: x={x:.2f}, linkage height={y:.3f}")
        print("Lower joins mean more similar frequency rows; high joins mean broader cluster merges.")

for trace in tree.data:
    trace.on_click(tree_clicked)
display(widgets.VBox([tree, click_out]))

    'data': [{'hovertemplate': 'linkage height %{y:.3f}<extra>Pipeline 3a</extra…

**Pipeline 3a:** The leaves are individual CWT frequency rows and the branch height is clustering distance. Tight low-height branches are similar frequency bands; high merges mean the algorithm is forcing less-similar bands together. Use this with Pipeline 2 to decide whether `n_subclusters`, `n_clusters`, or linkage settings are too coarse.

## Pipeline 4. Final Reconstruction And PCA Event Calls

This is the final output view. The denoised trace is shown here only, and event starts are consolidated from a single PCA event score rather than plotted directly from every cluster.

In [7]:
recon = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.58, 0.25, 0.17],
    subplot_titles=(
        "Pipeline 4a. Real input and final denoised output",
        "Pipeline 4b. PCA event score with threshold",
        "Pipeline 4c. Consolidated event calls vs annotated APs",
    ),
    vertical_spacing=0.10,
)
recon.add_trace(go.Scatter(x=t_plot, y=input_plot, mode="lines", name="z-scored input", line={"color": "#9ecae9", "width": 1}), row=1, col=1)
recon.add_trace(go.Scatter(x=t_plot, y=denoised_plot, mode="lines", name="denoised", line={"color": "#d62728", "width": 1.5}), row=1, col=1)
recon.add_trace(go.Scatter(x=recording.events_ap_times_s, y=np.interp(recording.events_ap_times_s, recording.t, denoiser_input), mode="markers", name="annotated AP", marker={"color": "#d7263d", "size": 5}), row=1, col=1)
if len(pca_event_indices):
    recon.add_trace(go.Scatter(x=pca_event_times, y=np.interp(pca_event_times, recording.t, denoised), mode="markers", name="PCA event call", marker={"color": "#111111", "size": 6}), row=1, col=1)

recon.add_trace(go.Scatter(x=t_plot, y=pca_score_plot, mode="lines", name="PCA event score", line={"color": "#1b998b", "width": 1.2}), row=2, col=1)
recon.add_trace(go.Scatter(x=[recording.t[0], recording.t[-1]], y=[pca_threshold_z, pca_threshold_z], mode="lines", name=f"threshold z={pca_threshold_z:.1f}", line={"color": "#111111", "dash": "dash", "width": 1}), row=2, col=1)
if len(pca_event_indices):
    recon.add_trace(go.Scatter(x=pca_event_times, y=pca_event_score_z[pca_event_indices], mode="markers", name="score peaks", marker={"color": "#111111", "size": 5}), row=2, col=1)
    recon.add_trace(go.Scatter(x=pca_event_times, y=np.ones_like(pca_event_times), mode="markers", name="PCA calls raster", marker={"color": "#111111", "size": 8, "symbol": "line-ns-open"}), row=3, col=1)
if len(recording.events_ap_times_s):
    recon.add_trace(go.Scatter(x=recording.events_ap_times_s, y=np.zeros_like(recording.events_ap_times_s), mode="markers", name="annotated AP raster", marker={"color": "#d7263d", "size": 8, "symbol": "line-ns-open"}), row=3, col=1)

recon.update_xaxes(title_text="time (s)", row=3, col=1)
recon.update_yaxes(title_text="z", row=1, col=1)
recon.update_yaxes(title_text="score z", row=2, col=1)
recon.update_yaxes(tickmode="array", tickvals=[0, 1], ticktext=["annotated AP", "PCA calls"], range=[-0.5, 1.5], row=3, col=1)
recon.update_layout(
    title={"text": "Pipeline 4. Final Reconstruction And PCA Event Calls", "x": 0.01, "xanchor": "left"},
    height=820,
    template="plotly_white",
    hovermode="x unified",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.03, "x": 0},
    margin={"t": 135, "b": 60, "l": 80, "r": 40},
)
recon.show()

**Pipeline 4a:** This is the first place to judge the final denoised answer. The blue trace is the real z-scored input, the red trace is the denoised output, red dots are the reference AP annotations, and black dots are the consolidated PCA event calls.

**Pipeline 4b:** This is the event thresholding view. The score is PC1 of the masked, scale-corrected cluster traces, sign-aligned to the input, smoothed over about 1.5 ms, robust-z-scored, then peak-picked above the dashed threshold with a 6 ms minimum distance. This is why the same peak should no longer be called multiple times.

**Pipeline 4c:** This raster compares final PCA calls to the annotated AP times. A black tick without a red tick can be a candidate subthreshold or false-positive event; a red tick without a nearby black tick means the current cluster selection or thresholding missed that AP.

## Pipeline 5. Template Matching

This step builds an AP-like template from the highest-amplitude PCA candidate events, then uses cosine similarity to keep only candidates with the right shape. The threshold is interactive so the rejection level can be tuned directly from the figure.

In [8]:
template_pre_ms = 4.0
template_post_ms = 10.0
template_align_ms = 3.0
template_high_amplitude_quantile = 0.75
template_default_threshold = 0.30


def build_event_template(
    signal,
    event_indices,
    fs_hz,
    pre_ms=4.0,
    post_ms=10.0,
    align_ms=3.0,
    high_amplitude_quantile=0.75,
):
    signal = np.asarray(signal, dtype=float)
    event_indices = np.asarray(event_indices, dtype=int)
    if event_indices.size == 0:
        raise ValueError("No PCA events are available for template matching.")

    pre = max(1, int(round((pre_ms / 1000.0) * fs_hz)))
    post = max(1, int(round((post_ms / 1000.0) * fs_hz)))
    align = max(1, int(round((align_ms / 1000.0) * fs_hz)))
    polarity = 1 if np.median(signal[event_indices]) >= 0 else -1

    snippets = []
    aligned_indices = []
    source_indices = []
    amplitudes = []
    for idx in event_indices:
        search_lo = max(0, idx - align)
        search_hi = min(len(signal), idx + align + 1)
        if search_hi <= search_lo:
            continue
        peak = search_lo + int(np.argmax(polarity * signal[search_lo:search_hi]))
        snippet_lo = peak - pre
        snippet_hi = peak + post + 1
        if snippet_lo < 0 or snippet_hi > len(signal):
            continue
        snippet = polarity * signal[snippet_lo:snippet_hi].copy()
        snippet -= np.median(snippet[:pre])
        snippets.append(snippet)
        aligned_indices.append(peak)
        source_indices.append(idx)
        amplitudes.append(snippet[pre])

    snippets = np.asarray(snippets, dtype=float)
    aligned_indices = np.asarray(aligned_indices, dtype=int)
    source_indices = np.asarray(source_indices, dtype=int)
    amplitudes = np.asarray(amplitudes, dtype=float)
    if snippets.size == 0:
        raise ValueError("No PCA events were far enough from the edges for template matching.")

    n_high = max(4, int(np.ceil(snippets.shape[0] * (1.0 - high_amplitude_quantile))))
    n_high = min(snippets.shape[0], n_high)
    high_order = np.argsort(amplitudes)[::-1]
    high_indices = np.sort(high_order[:n_high])
    high_snippets = snippets[high_indices]
    template = np.mean(high_snippets, axis=0)

    centered_snippets = snippets - np.mean(snippets, axis=1, keepdims=True)
    centered_template = template - np.mean(template)
    denom = np.linalg.norm(centered_snippets, axis=1) * np.linalg.norm(centered_template)
    cosine_scores = np.divide(
        centered_snippets @ centered_template,
        denom,
        out=np.zeros(snippets.shape[0]),
        where=denom > 0,
    )
    time_ms = (np.arange(snippets.shape[1]) - pre) / fs_hz * 1000.0
    return {
        "time_ms": time_ms,
        "snippets": snippets,
        "high_snippets": high_snippets,
        "template": template,
        "aligned_indices": aligned_indices,
        "source_indices": source_indices,
        "amplitudes": amplitudes,
        "high_indices": high_indices,
        "cosine_scores": cosine_scores,
        "polarity": polarity,
    }


template_result = build_event_template(
    denoised,
    pca_event_indices,
    recording.fs_hz,
    pre_ms=template_pre_ms,
    post_ms=template_post_ms,
    align_ms=template_align_ms,
    high_amplitude_quantile=template_high_amplitude_quantile,
)
template_times = template_result["aligned_indices"] / recording.fs_hz
template_scores = template_result["cosine_scores"]
template_initial_keep = template_scores >= template_default_threshold

print(
    f"Template built from {len(template_result['high_snippets'])} high-amplitude events | "
    f"cosine threshold {template_default_threshold:.2f} keeps "
    f"{int(template_initial_keep.sum())}/{len(template_scores)} PCA calls"
)

Template built from 10 high-amplitude events | cosine threshold 0.30 keeps 35/38 PCA calls


In [9]:
template_fig = go.FigureWidget(make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.46, 0.54],
    subplot_titles=(
        "Pipeline 5a. Template PSTH",
        "Pipeline 5b. Event Raster",
    ),
    horizontal_spacing=0.18,
))

for snippet in template_result["high_snippets"]:
    template_fig.add_trace(
        go.Scatter(
            x=template_result["time_ms"],
            y=snippet,
            mode="lines",
            line={"color": "#4c78a8", "width": 1},
            opacity=0.18,
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
template_fig.add_trace(
    go.Scatter(
        x=template_result["time_ms"],
        y=template_result["template"],
        mode="lines",
        name="template average",
        line={"color": "#111111", "width": 3},
        hovertemplate="%{x:.2f} ms<br>template %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

pca_raster = template_fig.add_trace(
    go.Scatter(
        x=pca_event_times,
        y=np.full_like(pca_event_times, 2.0),
        mode="markers",
        name="PCA calls",
        marker={"color": "#111111", "size": 8, "symbol": "line-ns-open"},
        hovertemplate="PCA call<br>%{x:.3f} s<extra></extra>",
    ),
    row=1,
    col=2,
)
annotation_raster = template_fig.add_trace(
    go.Scatter(
        x=recording.events_ap_times_s,
        y=np.ones_like(recording.events_ap_times_s),
        mode="markers",
        name="annotated AP",
        marker={"color": "#d7263d", "size": 8, "symbol": "line-ns-open"},
        hovertemplate="annotated AP<br>%{x:.3f} s<extra></extra>",
    ),
    row=1,
    col=2,
)
matched_trace = go.Scatter(
    x=[],
    y=[],
    mode="markers",
    name="template matched AP",
    marker={"color": "#1b998b", "size": 9, "symbol": "line-ns-open"},
    customdata=[],
    hovertemplate="template match<br>%{x:.3f} s<br>cosine %{customdata:.2f}<extra></extra>",
)
template_fig.add_trace(matched_trace, row=1, col=2)

template_fig.update_xaxes(title_text="time from aligned peak (ms)", row=1, col=1)
template_fig.update_xaxes(title_text="time (s)", row=1, col=2)
template_fig.update_yaxes(title_text="baseline-subtracted z", row=1, col=1)
template_fig.update_yaxes(
    tickmode="array",
    tickvals=[0, 1, 2],
    ticktext=["matched", "annotated", "PCA calls"],
    range=[-0.5, 2.5],
    side="right",
    row=1,
    col=2,
)
template_fig.update_layout(
    title={
        "text": "Pipeline 5. Template Matching Filter",
        "x": 0.01,
        "xanchor": "left",
        "font": {"size": 24},
    },
    height=640,
    template="plotly_white",
    hovermode="closest",
    font={"size": 14},
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "left",
        "x": 0,
    },
    margin={"t": 115, "b": 140, "l": 85, "r": 80},
)
for annotation in template_fig.layout.annotations:
    annotation.font.size = 15

threshold_slider = widgets.FloatSlider(
    value=template_default_threshold,
    min=-0.10,
    max=0.95,
    step=0.01,
    description="cosine",
    readout_format=".2f",
    continuous_update=True,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="360px"),
)
threshold_readout = widgets.HTML()


def update_template_threshold(*_):
    keep = template_scores >= threshold_slider.value
    matched_times = template_times[keep]
    matched_scores = template_scores[keep]
    with template_fig.batch_update():
        template_fig.data[-1].x = matched_times
        template_fig.data[-1].y = np.zeros_like(matched_times)
        template_fig.data[-1].customdata = matched_scores
    threshold_readout.value = (
        f"<b>{len(matched_times)}/{len(template_scores)}</b> PCA calls pass "
        f"cosine >= <b>{threshold_slider.value:.2f}</b>"
    )

threshold_slider.observe(update_template_threshold, names="value")
update_template_threshold()
controls = widgets.HBox(
    [threshold_slider, threshold_readout],
    layout=widgets.Layout(align_items="center", margin="0 0 12px 0"),
)
display(widgets.VBox([controls, template_fig]))

**Pipeline 5a:** The faint blue traces are the high-amplitude PCA candidate events used as ideal samples. The dark trace is their average and becomes the AP-shape template used for matching.

**Pipeline 5b:** The top row is every PCA event call, the middle row is the raw file's AP annotation, and the bottom row is the subset that passes template matching. Drag the cosine threshold to trade sensitivity for stricter AP-shape matching; higher thresholds should reject more PCA calls.